In [ ]:
!pip install -U langgraph langchain_groq langchain langchain-core
!pip install langchain_community faiss-cpu sentence-transformers
!pip install langchain-huggingface fpdf2 joblib pandas numpy

In [ ]:
"""
=============================================================================
MILESTONE 2 — INTELLIGENT LEARNING ANALYTICS: AGENTIC AI STUDY COACH
=============================================================================
Architecture: Supervisor-based routing + Predefined Mini Workflows
              + Strict State Control + Guarded Transitions

Industry Pattern Used:
  - Supervisor does INTENT CLASSIFICATION only (one LLM call)
  - Specialist nodes are DETERMINISTIC (no LLM routing inside)
  - State FLAGS prevent re-execution and loops
  - Hard ITERATION CAP as safety net
  - Node CONTRACTS enforced before execution
=============================================================================
"""

# ─────────────────────────────────────────────
# CELL 1 — INSTALLS
# ─────────────────────────────────────────────
# !pip install -U langgraph langchain_groq langchain langchain-core
# !pip install langchain_community faiss-cpu sentence-transformers
# !pip install langchain-huggingface fpdf2 joblib pandas numpy

# ─────────────────────────────────────────────
# CELL 2 — IMPORTS
# ─────────────────────────────────────────────
import os
import json
import operator
import numpy as np
import pandas as pd
import joblib
import faiss

from typing import Annotated, List, TypedDict, Literal, Optional
from pydantic import BaseModel, Field

from langchain_groq import ChatGroq
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from sentence_transformers import SentenceTransformer
from langgraph.graph import StateGraph, END

# ─────────────────────────────────────────────
# CELL 3 — LLM INITIALISATION
# ─────────────────────────────────────────────
# Set your Groq API key
os.environ["GROQ_API_KEY"] = "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"

llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0,
    max_retries=2,       # Retry on transient failures
    request_timeout=30   # Don't hang forever
)

# ─────────────────────────────────────────────
# CELL 4 — LOAD ML MODELS (Milestone 1 Artifacts)
# ─────────────────────────────────────────────
reg_model     = joblib.load('linear_model.pkl')
clf_model     = joblib.load('logistic_model.pkl')
cluster_model = joblib.load('kmeans_model.pkl')
scaler_reg    = joblib.load('scaler_reg.pkl')
scaler_clf    = joblib.load('scaler_clf.pkl')
scaler_cluster= joblib.load('scaler_cluster.pkl')

# ─────────────────────────────────────────────
# CELL 5 — DYNAMIC CLUSTER LABEL MAPPING
# This is the correct way — same as M1 app.py
# K-Means cluster IDs are random per training run.
# We always sort by ExamScore center to assign labels.
# ─────────────────────────────────────────────
_centers     = cluster_model.cluster_centers_
_exam_idx    = 0   # ExamScore is the first feature in cluster_cols
_sorted_ids  = np.argsort(_centers[:, _exam_idx])
CLUSTER_LABEL_MAP = {
    int(_sorted_ids[0]): "At-Risk",
    int(_sorted_ids[1]): "Average",
    int(_sorted_ids[2]): "High-Performer"
}
print(f"Cluster Label Map (dynamic): {CLUSTER_LABEL_MAP}")

# ─────────────────────────────────────────────
# CELL 6 — ML PIPELINE FUNCTION
# ─────────────────────────────────────────────
FEAT_COLS = [
    'MathScore', 'ReadingScore', 'WklyStudyHours', 'ParentEduc',
    'TestPrep_none', 'LunchType_standard', 'PracticeSport',
    'NrSiblings', 'Gender_male', 'IsFirstChild_yes', 'TransportMeans_school_bus'
]
CLUSTER_COLS = [
    'ExamScore', 'WklyStudyHours', 'ParentEduc',
    'LunchType_standard', 'TestPrep_none', 'PracticeSport'
]

# Safe defaults (medians from M1 dataset)
ML_DEFAULTS = {
    'math':          67.0,
    'reading':       70.0,
    'study_hours':   7.5,
    'siblings':      2.0,
    'parent_educ':   2,
    'sport':         1,
    'test_prep':     'none',
    'lunch':         'standard',
    'gender':        'female',
    'is_first_child':'yes',
    'transport':     'school_bus',
}

def run_ml_pipeline(user_data: dict) -> dict:
    """
    Takes a partial or full student profile dict.
    Merges with safe defaults so we never crash on missing keys.
    Returns predicted_score, status, category, confidence.
    """
    # Merge with defaults — user values override defaults
    d = {**ML_DEFAULTS, **user_data}

    features = {
        'MathScore':                 float(d['math']),
        'ReadingScore':              float(d['reading']),
        'WklyStudyHours':            float(d['study_hours']),
        'NrSiblings':                float(d['siblings']),
        'ParentEduc':                int(d['parent_educ']),
        'PracticeSport':             int(d['sport']),
        'TestPrep_none':             1 if str(d['test_prep']).lower() == 'none' else 0,
        'LunchType_standard':        1 if str(d['lunch']).lower() == 'standard' else 0,
        'Gender_male':               1 if str(d['gender']).lower() == 'male' else 0,
        'IsFirstChild_yes':          1 if str(d['is_first_child']).lower() == 'yes' else 0,
        'TransportMeans_school_bus': 1 if str(d['transport']).lower() == 'school_bus' else 0,
    }

    X = pd.DataFrame([features])[FEAT_COLS]

    # Regression
    pred_score = float(np.clip(reg_model.predict(scaler_reg.transform(X))[0], 0, 100))

    # Classification + confidence
    X_clf     = scaler_clf.transform(X)
    status    = clf_model.predict(X_clf)[0]
    proba     = clf_model.predict_proba(X_clf)[0]
    classes   = list(clf_model.classes_)
    conf      = round(float(proba[classes.index(status)]) * 100, 1)

    # Clustering (dynamic label map)
    cluster_feats = {
        'ExamScore':          pred_score,
        'WklyStudyHours':     features['WklyStudyHours'],
        'ParentEduc':         features['ParentEduc'],
        'LunchType_standard': features['LunchType_standard'],
        'TestPrep_none':      features['TestPrep_none'],
        'PracticeSport':      features['PracticeSport'],
    }
    X_cl  = pd.DataFrame([cluster_feats])[CLUSTER_COLS]
    c_id  = int(cluster_model.predict(scaler_cluster.transform(X_cl))[0])
    category = CLUSTER_LABEL_MAP.get(c_id, "Average")

    return {
        "predicted_score": round(pred_score, 2),
        "status":          status,
        "confidence":      conf,
        "category":        category,
        "pass_threshold":  69.0,
        "score_gap":       round(69.0 - pred_score, 2) if pred_score < 69.0 else 0.0
    }

# ─────────────────────────────────────────────
# CELL 7 — RAG: KNOWLEDGE BASE + FAISS INDEX
# ─────────────────────────────────────────────
# Expandable knowledge base — add more entries freely.
# Structure: {category/topic: [list of documents]}
KNOWLEDGE_BASE = [
    # ── At-Risk Strategies ──
    "At-Risk students should break study sessions into 20-minute blocks with 5-minute breaks (Pomodoro). Consistency beats marathon sessions.",
    "At-Risk students must prioritise fundamentals before attempting advanced problems. Gaps in basics cause cascading failures.",
    "At-Risk recovery tip: Focus on ONE weak subject per week rather than spreading effort thin across all subjects.",
    "At-Risk students benefit most from completing a test preparation course. Our data shows only 8% of At-Risk students completed test prep vs 100% of High-Performers.",
    "For At-Risk students scoring below 60: Start with chapter summaries, not full chapters. Build schema before detail.",

    # ── Average Level Strategies ──
    "Average students should attempt weekly mock tests to identify which topics cause the most errors.",
    "Average students can move to High-Performer by completing test preparation — it is the single biggest differentiator in our dataset.",
    "Average tip: Use spaced repetition for topics you already know, and active recall for new topics.",
    "Average students score between 68-69 on exam. A 1-hour daily increase in study time can push scores above the pass threshold.",

    # ── High-Performer Strategies ──
    "High-Performers should focus on timed mock exams to simulate real test conditions and improve speed.",
    "High-Performers: Peer teaching is the highest form of active recall. Teach concepts to others to deepen your own understanding.",
    "High-Performers should identify the 5% of their mistakes and turn those into their entire study focus.",

    # ── Math Strategies ──
    "Math tip: Always write down the formula before substituting values. This prevents arithmetic errors.",
    "Quadratic equations: Use the formula x = (-b ± √(b² - 4ac)) / 2a. Always check discriminant first.",
    "Math study strategy: Do 10 problems from yesterday's topic before starting today's new topic (interleaved practice).",
    "Algebra fundamentals: Isolate the variable by performing the same operation on both sides of the equation.",
    "For improving math scores, Khan Academy's Algebra and Pre-Calculus courses are highly recommended (khanacademy.org).",

    # ── Reading/Writing Strategies ──
    "Reading comprehension tip: Read questions BEFORE the passage so you know what to look for.",
    "Writing tip: Use the PEEL structure for paragraphs — Point, Evidence, Explanation, Link.",
    "Reading score improvement: Practice summarizing each paragraph in one sentence immediately after reading it.",
    "Vocabulary building: Learn 5 new words per day using flashcards. Connect each word to a sentence you create.",

    # ── Test Preparation ──
    "Test preparation courses improve scores significantly. In our dataset, 100% of High-Performers completed test prep.",
    "Test prep strategy: Simulate exam conditions — full silence, timer running, no notes. Do this at least twice per week.",
    "Test anxiety management: 4-7-8 breathing technique before exam: inhale 4 seconds, hold 7, exhale 8.",

    # ── Study Hours & Habits ──
    "Students studying more than 7.5 hours per week consistently outperform those studying less in our dataset.",
    "Study environment matters: Remove phone from the room during study sessions. Notifications reduce focus by 23%.",
    "Sleep is part of studying. 7-9 hours of sleep consolidates memory. Cramming the night before reduces performance.",

    # ── General Academic Advice ──
    "The biggest mistake students make is passive re-reading. Active recall (closing book, writing what you remember) is 3x more effective.",
    "Resources: Khan Academy (khanacademy.org), Coursera free audit, MIT OpenCourseWare (ocw.mit.edu), YouTube channels 3Blue1Brown for math.",
    "Learning gap diagnosis: If you can explain a concept in simple terms, you know it. If you cannot, you have a gap.",
]

# Build FAISS index
print("Building RAG index...")
_embed_model = SentenceTransformer("all-MiniLM-L6-v2")
_embeddings  = _embed_model.encode(KNOWLEDGE_BASE, show_progress_bar=False)
_dim         = _embeddings.shape[1]
_rag_index   = faiss.IndexFlatL2(_dim)
_rag_index.add(np.array(_embeddings).astype("float32"))
print(f"RAG index built: {len(KNOWLEDGE_BASE)} documents, {_dim}-dim vectors")

def rag_search(query: str, k: int = 3, category: str = "") -> List[str]:
    """Search the knowledge base. Prepend category to query for better retrieval."""
    enriched_query = f"{category} student: {query}" if category else query
    q_vec = _embed_model.encode([enriched_query]).astype("float32")
    distances, indices = _rag_index.search(q_vec, k)
    return [KNOWLEDGE_BASE[i] for i in indices[0] if i < len(KNOWLEDGE_BASE)]

# ─────────────────────────────────────────────
# CELL 8 — AGENT STATE
# Key design: FLAGS control flow, not LLM
# ─────────────────────────────────────────────
class AgentState(TypedDict):
    # ── Conversation ──
    messages:          Annotated[List[BaseMessage], operator.add]

    # ── Student Profile (persists across turns) ──
    student_data:      dict    # Raw extracted inputs
    ml_results:        dict    # Model outputs

    # ── RAG Context ──
    retrieved_docs:    List[str]

    # ── Quiz State ──
    quiz_questions:    List[dict]
    current_q_idx:     int
    quiz_score:        int
    quiz_active:       bool

    # ── Planner Output ──
    study_plan:        str

    # ── ROUTING ──
    next_node:         str       # Set by supervisor, read by conditional edges
    intent:            str       # "analyse" | "plan" | "retrieve" | "quiz" | "chat"

    # ── FLOW CONTROL FLAGS (The guards that prevent loops) ──
    analysis_done:     bool      # True after analyser runs for this turn
    retrieval_done:    bool      # True after retriever runs for this turn
    plan_done:         bool      # True after planner runs for this turn

    # ── SAFETY ──
    iteration_count:   int       # Hard cap counter
    error_count:       int       # Track consecutive errors

# ─────────────────────────────────────────────
# CELL 9 — CONSTANTS
# ─────────────────────────────────────────────
MAX_ITERATIONS  = 8    # Hard cap: no conversation turn can take more than 8 node hops
MAX_ERRORS      = 3    # After 3 consecutive errors, gracefully terminate

# ─────────────────────────────────────────────
# CELL 10 — PYDANTIC SCHEMAS
# ─────────────────────────────────────────────

class Intent(BaseModel):
    """Supervisor's classification of user intent."""
    primary: Literal["analyse", "plan", "retrieve", "quiz", "chat"] = Field(
        description="The single primary intent of the user's message."
    )
    has_scores:    bool = Field(description="True if the message contains numeric scores or academic data.")
    wants_plan:    bool = Field(description="True if the message explicitly requests a plan, schedule, or timetable.")
    wants_quiz:    bool = Field(description="True if the message explicitly asks for a test, quiz, or to be challenged.")
    wants_resource:bool = Field(description="True if the message asks for resources, links, tips, or how-to information.")
    reasoning:     str  = Field(description="One-sentence explanation of the classification.")


class StudentProfile(BaseModel):
    """Structured extraction of student academic data."""
    math:          Optional[float] = Field(None, description="Math score 0-100")
    reading:       Optional[float] = Field(None, description="Reading score 0-100")
    study_hours:   Optional[float] = Field(None, description="Weekly study hours as a number")
    parent_educ:   Optional[int]   = Field(None, description="1=high_school 2=some_college 3=associates 4=bachelors 5=masters")
    test_prep:     Optional[str]   = Field(None, description="'none' or 'completed'")
    lunch:         Optional[str]   = Field(None, description="'standard' or 'free/reduced'")
    sport:         Optional[int]   = Field(None, description="0=never 1=sometimes 2=regularly")
    gender:        Optional[str]   = Field(None, description="'male' or 'female'")
    is_first_child:Optional[str]   = Field(None, description="'yes' or 'no'")
    transport:     Optional[str]   = Field(None, description="'school_bus' or 'public'")
    siblings:      Optional[int]   = Field(None, description="Number of siblings 0-6")


class StudyDay(BaseModel):
    day:            str       = Field(description="e.g. Day 1 — Monday")
    topic:          str       = Field(description="Focus area")
    activities:     List[str] = Field(description="2-3 specific tasks")
    estimated_time: str       = Field(description="e.g. 1.5 hours")


class WeeklyPlan(BaseModel):
    title:   str            = Field(description="Plan title")
    days:    List[StudyDay] = Field(description="7 daily plans")
    summary: str            = Field(description="Closing motivational note with one actionable priority")


class QuizQuestion(BaseModel):
    question:       str       = Field(description="The question text")
    options:        List[str] = Field(description="Exactly 4 options labelled A, B, C, D")
    correct_answer: str       = Field(description="The letter of the correct option: A, B, C, or D")
    explanation:    str       = Field(description="Why this answer is correct, in one sentence")


class QuizSet(BaseModel):
    topic:     str             = Field(description="Topic of the quiz")
    questions: List[QuizQuestion] = Field(description="Exactly 3 quiz questions")

# ─────────────────────────────────────────────
# CELL 11 — SUPERVISOR NODE
#
# Industry pattern:
#   - LLM does ONE thing: classify intent
#   - Rules do routing: no LLM for transitions
#   - Flags prevent re-running completed nodes
#   - Iteration cap prevents infinite loops
# ─────────────────────────────────────────────
def supervisor_node(state: AgentState) -> dict:
    print(f"\n{'='*50}")
    print(f"NODE: SUPERVISOR | Iteration: {state.get('iteration_count', 0)}")

    # ── SAFETY CHECK 1: Hard iteration cap ──
    iteration = state.get("iteration_count", 0)
    if iteration >= MAX_ITERATIONS:
        print(f"SAFETY: Max iterations ({MAX_ITERATIONS}) reached. Forcing END.")
        return {
            "next_node":       "end",
            "iteration_count": iteration + 1,
        }

    # ── SAFETY CHECK 2: Error cap ──
    if state.get("error_count", 0) >= MAX_ERRORS:
        print("SAFETY: Too many errors. Forcing END.")
        return {"next_node": "end"}

    # ── PRIORITY 1: Active quiz — NEVER break a quiz session ──
    if state.get("quiz_active", False):
        print("SUPERVISOR → quizzer (quiz is active)")
        return {
            "next_node":       "quizzer",
            "iteration_count": iteration + 1,
        }

    # ── Get the latest human message ──
    human_msgs = [m for m in state["messages"] if isinstance(m, HumanMessage)]
    if not human_msgs:
        return {"next_node": "end", "iteration_count": iteration + 1}
    user_msg = human_msgs[-1].content

    # ── Get current flags ──
    analysis_done  = state.get("analysis_done",  False)
    retrieval_done = state.get("retrieval_done", False)
    plan_done      = state.get("plan_done",      False)

    # ── STEP A: Classify intent via LLM (ONE call only) ──
    try:
      raw = llm.invoke(f"""
      Return STRICT JSON only.

      {{
        "primary": "analyse|plan|retrieve|quiz|chat",
        "has_scores": true/false,
        "wants_plan": true/false,
        "wants_quiz": true/false,
        "wants_resource": true/false,
        "reasoning": "one line"
      }}

      RULES:
      - Booleans MUST be true/false (not strings)
      - No extra text

      MESSAGE: "{user_msg}"
      """)

      content = raw.content.strip()
      content = content.replace("```json", "").replace("```", "").strip()

      parsed = json.loads(content)

      # Force correct types (VERY IMPORTANT)
      parsed["has_scores"] = bool(parsed.get("has_scores"))
      parsed["wants_plan"] = bool(parsed.get("wants_plan"))
      parsed["wants_quiz"] = bool(parsed.get("wants_quiz"))
      parsed["wants_resource"] = bool(parsed.get("wants_resource"))

      intent_result = Intent(**parsed)

      print(f"SUPERVISOR Intent: {intent_result.primary} | {intent_result.reasoning}")

    except Exception as e:
      print(f"Intent parsing failed: {e}")

      # 🔥 FALLBACK (VERY IMPORTANT)
      text = user_msg.lower()

      intent_result = Intent(
      primary="plan" if "plan" in text else
              "quiz" if "quiz" in text else
              "retrieve" if "tip" in text or "improve" in text else
              "analyse" if any(x in text for x in ["score", "marks"]) else
              "chat",
      # has_scores=any(char.isdigit() for char in text),
      has_scores=any(c.isdigit() for c in text) and ("math" in text or "reading" in text),
      wants_plan="plan" in text,
      wants_quiz="quiz" in text,
      wants_resource="tip" in text or "improve" in text,
      reasoning="fallback"
      )

    # ── STEP B: Rule-based routing using intent + flags ──
    # Priority: analyse > retrieve > plan > quiz > chat
    # Flags prevent re-running a node that already completed this turn

    if intent_result.has_scores and not analysis_done:
        next_n = "analyser"

    elif intent_result.wants_plan and not plan_done:
        # Planner needs ML results — run analyser first if not done
        if not analysis_done and not state.get("ml_results"):
            next_n = "analyser"
        else:
            # Run retriever first if not done (planner uses RAG context)
            next_n = "retriever" if not retrieval_done else "planner"

    elif intent_result.wants_resource and not retrieval_done:
        next_n = "retriever"

    elif intent_result.wants_quiz:
        next_n = "quizzer"

    elif analysis_done and intent_result.wants_plan and not plan_done:
        next_n = "retriever" if not retrieval_done else "planner"

    elif analysis_done and retrieval_done and not plan_done and intent_result.wants_plan:
        next_n = "planner"

    else:
        # Everything requested is done, or it's a simple chat
        next_n = "end"

    print(f"SUPERVISOR → {next_n} (flags: analysis={analysis_done}, retrieval={retrieval_done}, plan={plan_done})")
    return {
        "next_node":       next_n,
        "intent":          intent_result.primary,
        "iteration_count": iteration + 1,
    }

# ─────────────────────────────────────────────
# CELL 12 — ANALYSER NODE
#
# Contract:
#   INPUT:  latest human message
#   OUTPUT: student_data (merged), ml_results, analysis_done=True
# ─────────────────────────────────────────────
def analyser_node(state: AgentState) -> dict:
    print("NODE: ANALYSER")

    human_msgs = [m for m in state["messages"] if isinstance(m, HumanMessage)]
    user_msg   = human_msgs[-1].content

    # ── Extract structured data from message ──
    try:
        extractor = llm.with_structured_output(StudentProfile)
        extracted: StudentProfile = extractor.invoke(
            f"""Extract student academic data from this message.
            Only extract values EXPLICITLY stated or clearly implied.
            Do NOT guess or fill in values not mentioned.

            MESSAGE: "{user_msg}"

            For study_hours: if they say "<5 hrs" use 2.5, "5-10 hrs" use 7.5, ">10 hrs" use 12.0
            For parent_educ: map to 1-5 scale if mentioned.
            For test_prep: if they say "completed/did test prep" = 'completed', else 'none'."""
        )
        # Filter None values — don't overwrite existing data with None
        new_data = {k: v for k, v in extracted.model_dump().items() if v is not None}
    except Exception as e:
        print(f"ANALYSER: Extraction error: {e}")
        new_data = {}

    # ── Merge with existing student_data (persistent across turns) ──
    current_data = state.get("student_data", {}).copy()
    current_data.update(new_data)  # New values override old ones

    # ── Run ML Pipeline ──
    try:
        ml_results = run_ml_pipeline(current_data)
    except Exception as e:
        print(f"ANALYSER: ML pipeline error: {e}")
        ml_results = state.get("ml_results", {
            "predicted_score": 0, "status": "Unknown",
            "category": "Unknown", "confidence": 0, "score_gap": 0
        })

    # ── Build analysis response ──
    score    = ml_results.get("predicted_score", "N/A")
    status   = ml_results.get("status", "N/A")
    category = ml_results.get("category", "N/A")
    conf     = ml_results.get("confidence", "N/A")
    gap      = ml_results.get("score_gap", 0)

    if new_data:
        captured = ", ".join(new_data.keys())
        if gap > 0:
            gap_msg = f" You are {gap} marks below the pass threshold of 69.0 — this is bridgeable."
        else:
            gap_msg = " You are above the pass threshold of 69.0."

        response = (
            f"Profile updated with: **{captured}**.\n\n"
            f"**Predicted Exam Score:** {score}/100 (Confidence: {conf}%)\n"
            f"**Pass/Fail Status:** {status}\n"
            f"**Learner Category:** {category}\n"
            f"{gap_msg}"
        )
    else:
        response = (
            f"I have your current profile on record.\n\n"
            f"**Predicted Exam Score:** {score}/100\n"
            f"**Status:** {status} | **Category:** {category}\n\n"
            f"Share your scores (math, reading) or study habits to get a more accurate prediction."
        )

    return {
        "student_data":  current_data,
        "ml_results":    ml_results,
        "analysis_done": True,
        "messages":      [AIMessage(content=response)],
    }

# ─────────────────────────────────────────────
# CELL 13 — RETRIEVER NODE
#
# Contract:
#   INPUT:  user query + ml_results (optional)
#   OUTPUT: retrieved_docs, retrieval_done=True
# ─────────────────────────────────────────────
def retriever_node(state: AgentState) -> dict:
    print("NODE: RETRIEVER")

    human_msgs = [m for m in state["messages"] if isinstance(m, HumanMessage)]
    user_query = human_msgs[-1].content
    category   = state.get("ml_results", {}).get("category", "")

    # ── RAG Search ──
    try:
        docs = rag_search(user_query, k=3, category=category)
    except Exception as e:
        print(f"RETRIEVER: RAG search error: {e}")
        docs = ["Focus on consistent daily practice and completing test preparation courses."]

    context = "\n".join([f"- {d}" for d in docs])

    # ── LLM Synthesis — grounded in retrieved docs only ──
    # If intent is 'plan', just store docs silently for planner to use
    # If intent is 'retrieve', give a full response
    intent = state.get("intent", "retrieve")
    plan_done = state.get("plan_done", False)

    if intent == "plan" and not plan_done:
        # Silent retrieval for planner — no user-facing message needed
        print("RETRIEVER: Silent mode (feeding planner)")
        return {
            "retrieved_docs":  docs,
            "retrieval_done":  True,
        }

    # Full response for resource requests
    try:
        system = SystemMessage(content="""You are an Academic Resource Specialist for a Student Analytics system.
        Your ONLY job is to synthesize the retrieved documents into helpful, actionable advice.
        RULES:
        - Use ONLY the provided documents as your source
        - Tailor the advice to the student's category if provided
        - Give maximum 3 specific, actionable points
        - If resources/links are in the docs, include them
        - End with one clear next action""")

        human = HumanMessage(content=f"""
        Student Category: {category if category else 'General'}
        Student Question: {user_query}

        Retrieved Knowledge:
        {context}

        Provide concise, actionable advice based on the retrieved knowledge.""")

        response = llm.invoke([system, human])
        reply = response.content
    except Exception as e:
        print(f"RETRIEVER: LLM synthesis error: {e}")
        reply = f"Here are some relevant strategies:\n{context}"

    return {
        "retrieved_docs": docs,
        "retrieval_done": True,
        "messages":       [AIMessage(content=reply)],
    }

# ─────────────────────────────────────────────
# CELL 14 — PLANNER NODE
#
# Contract:
#   INPUT:  ml_results (required), retrieved_docs (optional)
#   OUTPUT: study_plan (markdown), plan_done=True
# ─────────────────────────────────────────────
def planner_node(state: AgentState) -> dict:
    print("NODE: PLANNER")

    ml_results = state.get("ml_results", {})
    category   = ml_results.get("category", "Unknown")
    score      = ml_results.get("predicted_score", "N/A")
    status     = ml_results.get("status", "N/A")
    gap        = ml_results.get("score_gap", 0)

    # Get RAG context if available
    retrieved_docs = state.get("retrieved_docs", [])
    rag_context    = "\n".join([f"- {d}" for d in retrieved_docs]) if retrieved_docs else ""

    # Get user's specific request
    human_msgs   = [m for m in state["messages"] if isinstance(m, HumanMessage)]
    user_request = human_msgs[-1].content if human_msgs else "Create a study plan"

    # ── Generate Structured Plan via LLM ──
    try:
        planner_llm = llm.with_structured_output(WeeklyPlan)

        category_instructions = {
            "At-Risk": (
                "Focus on FUNDAMENTALS only. Short 20-30 min sessions. "
                "Daily consistency over long sessions. Confidence-building activities. "
                f"Priority: bridge the {gap} mark gap to reach the pass threshold."
            ),
            "Average": (
                "Mix of practice and concept refinement. "
                "Mock tests mid-week to diagnose weak spots. "
                "Strong push toward completing test preparation. "
                "Goal: move from Average to High-Performer."
            ),
            "High-Performer": (
                "Full timed mock exams. Advanced problem sets. "
                "Focus on the 5% of mistakes. Peer teaching for consolidation. "
                "Competitive-level materials."
            ),
            "Unknown": (
                "Balanced approach covering all subjects equally. "
                "Ask the student for scores to personalise further."
            )
        }

        instructions = category_instructions.get(category, category_instructions["Unknown"])

        prompt = f"""You are an expert Academic Study Planner.

STUDENT CONTEXT:
- Predicted Exam Score: {score}/100
- Status: {status}
- Learner Category: {category}
- Student Request: {user_request}

CATEGORY INSTRUCTIONS:
{instructions}

RELEVANT STUDY STRATEGIES FROM KNOWLEDGE BASE:
{rag_context if rag_context else "Use general best practices."}

GENERATE a structured 7-day study plan.
Each day must have a clear topic, 2-3 specific activities, and time estimate.
The summary must end with ONE clear highest-priority action for this student.
{"DO NOT add a sentence asking for scores since we already have them." if category != "Unknown" else ""}"""

        plan_obj: WeeklyPlan = planner_llm.invoke(prompt)

        # ── Render to Markdown ──
        md = f"### {plan_obj.title}\n"
        md += f"*Personalised for: **{category}** | Predicted Score: {score}/100 | Status: {status}*\n\n"
        md += "---\n"
        for d in plan_obj.days:
            md += f"**{d.day} — {d.topic}**\n"
            for act in d.activities:
                md += f"- {act}\n"
            md += f"*Estimated time: {d.estimated_time}*\n\n"
        md += "---\n"
        md += f"**Priority Note:** {plan_obj.summary}"

    except Exception as e:
        print(f"PLANNER: Generation error: {e}")
        md = (
            f"### 7-Day Study Plan ({category})\n\n"
            f"I encountered an issue generating your full plan. "
            f"Here's your key priority: Since you are in the **{category}** category with a "
            f"predicted score of **{score}**, focus on completing test preparation and "
            f"increasing study consistency to 7+ hours per week."
        )

    return {
        "study_plan": md,
        "plan_done":  True,
        "messages":   [AIMessage(content=md)],
    }

# ─────────────────────────────────────────────
# CELL 15 — QUIZZER NODE
#
# Three phases: INIT → EVALUATE → NEXT/END
# LLM answer grading handles A/B/C/D + full text answers
# ─────────────────────────────────────────────
def quizzer_node(state: AgentState) -> dict:
    print("NODE: QUIZZER")

    messages    = state.get("messages", [])
    quiz_active = state.get("quiz_active", False)
    questions   = state.get("quiz_questions", [])
    current_idx = state.get("current_q_idx", 0)
    score       = state.get("quiz_score", 0)
    category    = state.get("ml_results", {}).get("category", "General")

    last_human  = [m for m in messages if isinstance(m, HumanMessage)]
    user_msg    = last_human[-1].content if last_human else ""

    # ══════════════════════════════════════════
    # PHASE A: INITIALISE — Generate quiz
    # ══════════════════════════════════════════
    if not quiz_active or not questions:
        print("QUIZZER: Initialising new quiz...")

        try:
            raw = llm.invoke(f""" ... """)

            content = raw.content.strip()
            content = content.replace("```json", "").replace("```", "").strip()

            quiz_obj = json.loads(content)
            if "questions" not in quiz_obj or len(quiz_obj["questions"]) < 3:
                raise ValueError("Invalid quiz output")

            if "questions" not in quiz_obj or len(quiz_obj["questions"]) == 0:
                raise ValueError("Invalid quiz output")

        except Exception as e:
            print(f"QUIZZER: Init error → fallback: {e}")

            quiz_obj = {
                "topic": "Basic Algebra",
                "questions": [
                    {
                        "question": "2x + 5 = 11, x = ?",
                        "options": ["A) 2", "B) 3", "C) 4", "D) 5"],
                        "correct_answer": "B",
                        "explanation": "2x=6 → x=3"
                    },
                    {
                        "question": "3x = 9, x = ?",
                        "options": ["A) 2", "B) 3", "C) 4", "D) 5"],
                        "correct_answer": "B",
                        "explanation": "x = 3"
                    },
                    {
                        "question": "x + 4 = 10, x = ?",
                        "options": ["A) 4", "B) 5", "C) 6", "D) 7"],
                        "correct_answer": "C",
                        "explanation": "x = 6"
                    }
                ]
            }

        q0 = quiz_obj["questions"][0]
        options_text = "\n".join(q0["options"])

        reply = (
            f"Quiz started! Topic: **{quiz_obj['topic']}**\n\n"
            f"**Question 1 of {len(quiz_obj['questions'])}:**\n{q0['question']}\n\n"
            f"{options_text}\n\n"
            f"*Reply with A, B, C, or D*"
        )

        return {
            "quiz_questions": quiz_obj["questions"],
            "current_q_idx":  0,
            "quiz_score":     0,
            "quiz_active":    True,
            "messages":       [AIMessage(content=reply)],
        }

    # ══════════════════════════════════════════
    # PHASE B: EVALUATE answer + ADVANCE
    # ══════════════════════════════════════════
    current_q = questions[current_idx]

    # Smart answer grading — handles "A", "a", "option A", full text answers
    user_ans   = user_msg.strip().upper()
    correct    = current_q["correct_answer"].strip().upper()

    # Extract just the letter if user typed the full option text
    if len(user_ans) > 1:
        # Check if any option letter appears at start of answer
        for letter in ["A", "B", "C", "D"]:
            if user_ans.startswith(letter):
                user_ans = letter
                break
        else:
            # Check if answer text matches any option
            for i, opt in enumerate(current_q["options"]):
                if user_ans in opt.upper():
                    user_ans = ["A","B","C","D"][i]
                    break

    is_correct = (user_ans == correct)
    new_score  = score + 1 if is_correct else score

    if is_correct:
        feedback = f"✅ **Correct!**\n{current_q['explanation']}"
    else:
        feedback = f"❌ **Incorrect.** The correct answer was **{correct}**.\n{current_q['explanation']}"

    # ── Next question or end ──
    next_idx = current_idx + 1

    if next_idx < len(questions):
        next_q       = questions[next_idx]
        options_text = "\n".join(next_q["options"])
        reply = (
            f"{feedback}\n\n"
            f"---\n"
            f"**Question {next_idx + 1} of {len(questions)}:**\n{next_q['question']}\n\n"
            f"{options_text}\n\n"
            f"*Reply with A, B, C, or D*"
        )
        return {
            "current_q_idx": next_idx,
            "quiz_score":    new_score,
            "messages":      [AIMessage(content=reply)],
        }

    else:
        # ── Quiz finished ──
        total     = len(questions)
        pct       = round((new_score / total) * 100)
        perf_msg  = (
            "Excellent work! 🏆" if pct >= 80 else
            "Good effort! Keep practicing." if pct >= 50 else
            "Keep going — focus on the concepts you missed."
        )
        final_msg = (
            f"{feedback}\n\n"
            f"---\n"
            f"🏆 **Quiz Complete!**\n"
            f"**Score: {new_score}/{total} ({pct}%)**\n\n"
            f"{perf_msg}\n\n"
            f"Would you like a study plan targeting your weak areas, or more practice?"
        )
        return {
            "quiz_active":  False,
            "quiz_score":   new_score,
            "messages":     [AIMessage(content=final_msg)],
        }

# ─────────────────────────────────────────────
# CELL 16 — END NODE
# Handles: greetings, off-topic, thank-you, closing
# ─────────────────────────────────────────────
SYSTEM_PROMPT = """You are an intelligent, encouraging AI Study Coach for a student analytics system.
You help students understand their academic performance, create study plans, and prepare for exams.

YOUR PERSONALITY:
- Warm, encouraging, never discouraging
- Data-driven (reference the student's actual scores when available)
- Concise (max 3 sentences for general chat)
- Always end with one clear suggestion or question

YOUR BOUNDARIES:
- You only discuss academics, study strategies, and student performance
- If asked something off-topic, politely redirect: "I'm specialised in helping you with studies and exam preparation. What academic topic can I help with?"
- Never invent scores, statistics, or resource links not in your knowledge base"""

def end_node(state: AgentState) -> dict:
    print("NODE: END/CHAT")

    human_msgs = [m for m in state["messages"] if isinstance(m, HumanMessage)]
    user_msg   = human_msgs[-1].content if human_msgs else "Hello"
    ml_results = state.get("ml_results", {})
    category   = ml_results.get("category", "")

    # Build context-aware prompt
    profile_context = ""
    if ml_results:
        profile_context = (
            f"Student profile: Score={ml_results.get('predicted_score','?')}, "
            f"Category={category}, Status={ml_results.get('status','?')}"
        )

    try:
        response = llm.invoke([
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=f"{profile_context}\n\nStudent says: {user_msg}")
        ])
        reply = response.content
    except Exception as e:
        print(f"END: LLM error: {e}")
        reply = "I'm here to help with your studies! Share your scores or ask me anything about exam preparation."

    # Reset per-turn flags for the NEXT turn
    return {
        "messages":      [AIMessage(content=reply)],
        "analysis_done": False,   # Reset flags for next conversation turn
        "retrieval_done":False,
        "plan_done":     False,
        "iteration_count": 0,     # Reset iteration counter for next turn
    }

# ─────────────────────────────────────────────
# CELL 17 — BUILD THE GRAPH
#
# Architecture:
#   Supervisor (router) → Specialist nodes → END
#   Each specialist routes BACK to supervisor
#   Supervisor checks flags before re-routing
# ─────────────────────────────────────────────
def build_graph():
    workflow = StateGraph(AgentState)

    # Add nodes
    workflow.add_node("supervisor", supervisor_node)
    workflow.add_node("analyser",   analyser_node)
    workflow.add_node("retriever",  retriever_node)
    workflow.add_node("planner",    planner_node)
    workflow.add_node("quizzer",    quizzer_node)
    workflow.add_node("end",        end_node)

    # Entry point
    workflow.set_entry_point("supervisor")

    # Supervisor → specialist (conditional on next_node)
    workflow.add_conditional_edges(
        "supervisor",
        lambda s: s["next_node"],
        {
            "analyser":  "analyser",
            "retriever": "retriever",
            "planner":   "planner",
            "quizzer":   "quizzer",
            "end":       "end",
        }
    )

    # Specialists → supervisor (for multi-step flows)
    # e.g. analyse → supervisor → retriever → supervisor → planner → supervisor → end
    workflow.add_edge("analyser",  "supervisor")
    workflow.add_edge("retriever", "supervisor")
    workflow.add_edge("planner",   "supervisor")
    workflow.add_edge("quizzer",   "supervisor")

    # End → actual END
    workflow.add_edge("end", END)

    return workflow.compile()


graph = build_graph()
print("Graph compiled successfully!")

# ─────────────────────────────────────────────
# CELL 18 — INITIALISE STATE
# Call this at the START of each new chat session
# ─────────────────────────────────────────────
def get_initial_state() -> AgentState:
    return {
        "messages":       [],
        "student_data":   {},
        "ml_results":     {},
        "retrieved_docs": [],
        "quiz_questions": [],
        "current_q_idx":  0,
        "quiz_score":     0,
        "quiz_active":    False,
        "study_plan":     "",
        "next_node":      "supervisor",
        "intent":         "chat",
        "analysis_done":  False,
        "retrieval_done": False,
        "plan_done":      False,
        "iteration_count":0,
        "error_count":    0,
    }

# ─────────────────────────────────────────────
# CELL 19 — CHAT FUNCTION
# Maintains state across turns (session memory)
# ─────────────────────────────────────────────

# Global session state — persists across turns in the same session
_session_state = get_initial_state()

def chat(user_input: str) -> str:
    """
    Send a message to the agent. State persists across calls.
    Returns the agent's final response for this turn.

    Usage:
        reply = chat("I scored 55 in math and 60 in reading")
        reply = chat("Give me a study plan")
        reply = chat("Quiz me on algebra")
    """
    global _session_state

    # ── Reset per-turn flags (but keep student_data and ml_results) ──
    _session_state["analysis_done"]  = False
    _session_state["retrieval_done"] = False
    _session_state["plan_done"]      = False
    _session_state["iteration_count"]= 0

    # ── Add user message ──
    _session_state["messages"].append(HumanMessage(content=user_input))

    # ── Run graph ──
    try:
        result = graph.invoke(_session_state)
        _session_state = result
    except Exception as e:
        print(f"GRAPH ERROR: {e}")
        _session_state["messages"].append(
            AIMessage(content="I encountered an issue. Could you rephrase your question?")
        )
        return "I encountered an issue. Could you rephrase your question?"

    # ── Extract last AI message ──
    ai_msgs = [m for m in _session_state["messages"] if isinstance(m, AIMessage)]
    if ai_msgs:
        return ai_msgs[-1].content
    return "I'm here to help! Share your scores or ask me anything."


def reset_session():
    """Call this to start a completely fresh session."""
    global _session_state
    _session_state = get_initial_state()
    print("Session reset.")

# # ─────────────────────────────────────────────
# # CELL 20 — TEST RUNS
# # ─────────────────────────────────────────────
# if __name__ == "__main__":
#     reset_session()

#     # Test 1: Scores + plan request in one message
#     print("\n" + "="*60)
#     print("USER: I scored 55 in math and 62 in reading. Can you create a study plan?")
#     print("AGENT:", chat("I scored 55 in math and 62 in reading. Can you create a study plan?"))

#     # Test 2: Follow-up resource request
#     print("\n" + "="*60)
#     print("USER: Can you give me tips to improve my math?")
#     print("AGENT:", chat("Can you give me tips to improve my math?"))

#     # Test 3: Quiz
#     print("\n" + "="*60)
#     print("USER: Quiz me on algebra")
#     print("AGENT:", chat("Quiz me on algebra"))

#     # Test 4: Answer the quiz
#     print("\n" + "="*60)
#     print("USER: A")
#     print("AGENT:", chat("A"))



Cluster Label Map (dynamic): {2: 'At-Risk', 1: 'Average', 0: 'High-Performer'}
Building RAG index...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RAG index built: 30 documents, 384-dim vectors
Graph compiled successfully!


In [ ]:
# ─────────────────────────────────────────────
# CELL 20 — SINGLE COMPREHENSIVE TEST
# ─────────────────────────────────────────────
if __name__ == "__main__":
    reset_session()

    print("\n" + "="*70)
    print("TEST: FULL ORCHESTRATION FLOW")
    print("="*70)

    user_input = "I got 80 in math and 78 in reading. Just analyse my performance. Do not make a plan."

    print("\nUSER INPUT:")
    print(user_input)

    print("\n--- EXECUTION START ---\n")

    response = chat(user_input)

    print("\n--- FINAL RESPONSE ---\n")
    print(response)

    print("\n--- FINAL STATE DEBUG ---\n")
    print(f"Analysis Done   : {_session_state.get('analysis_done')}")
    print(f"Retrieval Done  : {_session_state.get('retrieval_done')}")
    print(f"Plan Done       : {_session_state.get('plan_done')}")
    print(f"Iteration Count : {_session_state.get('iteration_count')}")
    print(f"Intent Detected : {_session_state.get('intent')}")

Session reset.

TEST: FULL ORCHESTRATION FLOW

USER INPUT:
I got 80 in math and 78 in reading. Just analyse my performance. Do not make a plan.

--- EXECUTION START ---


NODE: SUPERVISOR | Iteration: 0
SUPERVISOR Intent: analyse | User wants analysis of their performance
SUPERVISOR → analyser (flags: analysis=False, retrieval=False, plan=False)
NODE: ANALYSER

NODE: SUPERVISOR | Iteration: 1
SUPERVISOR Intent: analyse | User wants analysis of their performance
SUPERVISOR → end (flags: analysis=True, retrieval=False, plan=False)
NODE: END/CHAT

--- FINAL RESPONSE ---

Your overall score of 79.4 indicates you're performing at an average level, and it's great that you've passed. Breaking down your scores, you've shown a slight strength in math with a score of 80, while your reading score is 2 points lower at 78. What do you think might have contributed to this small difference in scores between math and reading?

--- FINAL STATE DEBUG ---

Analysis Done   : False
Retrieval Done  : False


In [ ]:
# ─────────────────────────────────────────────
# INTERACTIVE CHAT SESSION
# ─────────────────────────────────────────────
if __name__ == "__main__":
    reset_session()

    print("\n🎓 AI Study Coach (type 'exit' to quit)\n")

    while True:
        user_input = input("You: ")

        if user_input.lower() in ["exit", "quit"]:
            print("Session ended.")
            break

        print("\n--- PROCESSING ---\n")

        response = chat(user_input)

        print("AI:", response)
        print("\n" + "-"*50 + "\n")

Session reset.

🎓 AI Study Coach (type 'exit' to quit)

You: hi

--- PROCESSING ---


NODE: SUPERVISOR | Iteration: 0
SUPERVISOR Intent: chat | User initiated conversation
SUPERVISOR → end (flags: analysis=False, retrieval=False, plan=False)
NODE: END/CHAT
AI: It's great to connect with you. I'm here to help you understand your academic performance and create a study plan tailored to your needs. What subject or topic would you like to focus on improving today?

--------------------------------------------------

You: can you create a study plan for me to excel in mathamatics

--- PROCESSING ---


NODE: SUPERVISOR | Iteration: 0
SUPERVISOR Intent: plan | user requested a study plan
SUPERVISOR → analyser (flags: analysis=False, retrieval=False, plan=False)
NODE: ANALYSER

NODE: SUPERVISOR | Iteration: 1
SUPERVISOR Intent: plan | user requested a study plan
SUPERVISOR → retriever (flags: analysis=True, retrieval=False, plan=False)
NODE: RETRIEVER
RETRIEVER: Silent mode (feeding planner)

